# Réseau de neurones simple avec PyTorch Lightning

Dans ces travaux pratiques, nous allons voir comment utiliser PyTorch Lightning pour construire des réseaux de neurones simples.

Il n'est pas nécessaire d'utiliser une carte graphique fournie par Colab pour ce TP, tout est rapide sur CPU. Vous pouvez donc changer d'environnement si celui qui vous a été attribué a une carte graphique (`Exécution > Modifier le type d'exécution`).

In [ ]:
!pip install -q lightning torchmetrics optuna tensorboard
import datetime

import lightning
import optuna
import torch
import torchmetrics
import torchvision
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

## Récupération des données

Cette fois nous allons récupérer le dataset par torchvision.

In [ ]:
train_data = torchvision.datasets.MNIST("data", train=True, download=True)
test_data = torchvision.datasets.MNIST("data", train=False, download=True)
X_train, y_train = train_data.data, train_data.targets
X_test, y_test = test_data.data, test_data.targets

## Regardons les données

Dans les cellules suivantes, étudiez comment sont stockées les données.

In [ ]:
# Votre code ici

### Solution

In [ ]:
print(f"Format de X_train : {tuple(X_train.shape)}")
print(f"Format de X_test : {tuple(X_test.shape)}")
print(f"Format de y_train : {tuple(y_train.shape)}")
print(f"Format de y_test : {tuple(y_test.shape)}")

In [ ]:
print(f"X_train type : {X_train.dtype}")
print(f"y_train type : {y_train[0].dtype}")
print(f"y_train exemple : {y_train[0]}")

In [ ]:
import matplotlib.pyplot as plt


n = 10
f, ax = plt.subplots(1, n, figsize=(n * 1.4, 2))
for i in range(n):
    ax[i].imshow(X_train[i], cmap="gray_r")
    ax[i].set_title(int(y_train[i]))
    ax[i].axis("off")
plt.show()

## Transformation des données

Nous devons effectuer quelques transformations sur ces données :

1. On pourrait conserver les formes originales des tenseurs `(_, 28, 28)` mais il sera plus aisé de travailler sur des tenseurs de forme `(_, 28²)` où `_` est le nombre original d'exemples.
2. On souhaite utiliser l'intervalle $[0, 1]$ plutôt que des valeurs entre $0$ et $255$ pour nos images. Transformez les entrées pour cela.
3. Nous allons voir deux manières de définir la fonction de perte. La première nécessite que les classes soit [one-hot encodées](https://fr.wikipedia.org/wiki/Encodage_one-hot).

Quelques fonctions qui peuvent s'avérer utiles :

- [`torch.Tensor.reshape`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.reshape.html)
- [`torch.nn.functional.one_hot`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.one_hot.html)

*Effectuez les deux premières transformations sur `X_train` et `X_test` et la dernière sur `y_train` et `y_test`.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
nb_classes = 10
input_dim = 28 * 28

# On veut mettre X « à plat » pour que l'input de notre réseau soit un vecteur
# de taille input_dim
X_train = X_train.reshape(X_train.shape[0], input_dim)
#syntaxe  équivalente
X_train = X_train.reshape(-1, input_dim)

X_test = X_test.reshape(X_test.shape[0], input_dim)
#syntaxe équivalente
X_test = X_test.reshape(-1, input_dim)

# Transformation vers [0, 1]
X_train = X_train / 255.0
X_test = X_test / 255.0

# Conversion des classes en vecteurs one-hot
Y_train = nn.functional.one_hot(y_train, nb_classes).float()
Y_test = nn.functional.one_hot(y_test, nb_classes).float()

## Création d'un modèle

Créez un modèle sans couche cachée qui prend en input une image et qui essaye de prédire la classe correspondante.

Affichez un résumé de ce modèle et expliquez les nombres que vous voyez.

Le `LightningModule` ci-dessous se charge de la fonction de perte, de la métrique et de l'optimiseur : le modèle qu'on lui donne renvoie des logits, l'entropie croisée acceptant aussi bien des indices de classe que des vecteurs one-hot comme cibles.


In [ ]:
class MNISTClassifier(lightning.LightningModule):
  """Lightning wrapper: cross-entropy on the logits, accuracy as a metric."""

  def __init__(self,
               model: nn.Module,
               optimizer: type[torch.optim.Optimizer] = torch.optim.SGD,
               learning_rate: float = 0.01,
               weight_decay: float = 0.,
               num_classes: int = nb_classes) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model
    # Une entrée d'exemple permet à Lightning d'afficher la forme des tenseurs
    # d'entrée et de sortie de chaque couche dans le résumé du modèle
    self.example_input_array = torch.zeros(1, input_dim)
    # torchmetrics demande une instance de métrique par étape
    self.accuracies = nn.ModuleDict({
        f"{stage}_accuracy": torchmetrics.Accuracy(task="multiclass",
                                                   num_classes=num_classes)
        for stage in ("train", "val", "test")
    })

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.model(images)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor],
            stage: str) -> torch.Tensor:
    images, targets = batch
    logits = self(images)
    loss = nn.functional.cross_entropy(logits, targets)
    # La métrique, elle, a besoin d'indices de classe
    labels = targets.argmax(dim=-1) if targets.dim() > 1 else targets
    accuracy = self.accuracies[f"{stage}_accuracy"]
    accuracy(logits, labels)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True,
             prog_bar=True)
    self.log(f"{stage}_accuracy", accuracy, on_step=False, on_epoch=True,
             prog_bar=True)
    return loss

  def training_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                    batch_index: int) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                      batch_index: int) -> torch.Tensor:
    return self._step(batch, "val")

  def test_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                batch_index: int) -> torch.Tensor:
    return self._step(batch, "test")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return self.hparams.optimizer(self.parameters(),
                                  lr=self.hparams.learning_rate,
                                  weight_decay=self.hparams.weight_decay)


def get_dataloaders(X: torch.Tensor,
                    Y: torch.Tensor,
                    batch_size: int = 128,
                    validation_split: float = 0.2
                    ) -> tuple[DataLoader, DataLoader]:
  """Sépare les données entre entraînement et validation."""
  train_dataset, val_dataset = random_split(
      TensorDataset(X, Y), [1 - validation_split, validation_split])
  return (DataLoader(train_dataset, batch_size=batch_size, shuffle=True),
          DataLoader(val_dataset, batch_size=batch_size))

In [ ]:
# model = nn.Sequential(???)
# classifier = MNISTClassifier(model)
# print(ModelSummary(classifier, max_depth=-1))

### Solution

In [ ]:
model = nn.Sequential(
    nn.Linear(input_dim, nb_classes))
classifier = MNISTClassifier(model)
print(ModelSummary(classifier, max_depth=-1))

## Apprentissage

Effectuez un apprentissage de ce modèle sur les données.

On utilisera :

- 128 comme taille de batch
- 10 itérations
- 20% de la base de train comme base de validation

In [ ]:
# Votre code ici

### Solution

In [ ]:
# Le Trainer utilise le LightningModule dans une boucle d'entraînement complète
train_loader, val_loader = get_dataloaders(X_train, Y_train, batch_size=128)

trainer = lightning.Trainer(max_epochs=10,
                            accelerator="auto",
                            devices=1,
                            logger=CSVLogger("logs", name="simple"),
                            enable_checkpointing=False)
trainer.fit(classifier, train_loader, val_loader)


def evaluate(trainer: lightning.Trainer,
             classifier: lightning.LightningModule,
             one_hot: bool) -> None:
  test_loader = DataLoader(TensorDataset(X_test,
                                         Y_test if one_hot else y_test),
                           batch_size=128)
  score = trainer.test(classifier, test_loader, verbose=False)[0]
  print(f"Perte sur le test : {score['test_loss']}")
  print(f"Accuracy sur le test : {score['test_accuracy']}")


evaluate(trainer, classifier, True)

Solution alternative qui utilise directement `y_train` plutôt que `Y_train` comme cible de l'entropie croisée :

In [ ]:
model2 = nn.Sequential(
    nn.Linear(input_dim, nb_classes))
classifier2 = MNISTClassifier(model2)
print(ModelSummary(classifier2, max_depth=-1))

train_loader2, val_loader2 = get_dataloaders(X_train, y_train, batch_size=128)

trainer2 = lightning.Trainer(max_epochs=10,
                             accelerator="auto",
                             devices=1,
                             logger=CSVLogger("logs", name="simple_sparse"),
                             enable_checkpointing=False)
trainer2.fit(classifier2, train_loader2, val_loader2)

evaluate(trainer2, classifier2, one_hot=False)

Solution avec 4 couches cachés de taille 20, une régularisation L2 des paramètres ainsi qu'une initialisation orthogonale des matrices de poids

In [ ]:
def dense(in_features: int, out_features: int, activation: bool = True
          ) -> nn.Module:
  layer = nn.Linear(in_features, out_features)
  nn.init.orthogonal_(layer.weight)
  return nn.Sequential(layer, nn.ReLU()) if activation else layer


def build_deep_model() -> nn.Module:
  # La régularisation L2 des poids et des biais est portée par l'optimiseur,
  # sous le nom de weight decay
  return nn.Sequential(
      dense(input_dim, 20),
      dense(20, 20),
      dense(20, 20),
      dense(20, 20),
      dense(20, nb_classes, activation=False))


deep_classifier = MNISTClassifier(build_deep_model(),
                                  optimizer=torch.optim.Adam,
                                  learning_rate=1e-3,
                                  weight_decay=0.01)
print(ModelSummary(deep_classifier, max_depth=-1))

train_loader, val_loader = get_dataloaders(X_train, y_train, batch_size=128)

deep_trainer = lightning.Trainer(max_epochs=10,
                                 accelerator="auto",
                                 devices=1,
                                 logger=CSVLogger("logs", name="deep"),
                                 enable_checkpointing=False)
deep_trainer.fit(deep_classifier, train_loader, val_loader)
evaluate(deep_trainer, deep_classifier, one_hot=False)

## Recherche d'hyper-paramètres

Pour trouver les hyper-paramètres optimaux, il est possible d'utiliser la librairie [`optuna`](https://optuna.org/). Pour cela, il faut définir une fonction objectif qui échantillonne les paramètres, entraîne le modèle correspondant et renvoie la métrique à optimiser. Référez-vous à l'exemple de la page d'accueil d'Optuna pour définir une telle fonction puis utilisez [l'échantillonneur bayésien TPE](https://optuna.readthedocs.io/en/stable/reference/samplers/generated/optuna.samplers.TPESampler.html), qui est celui par défaut, pour trouver les hyper-paramètres optimaux de votre modèle.

In [ ]:
# def objective(trial: optuna.Trial) -> float:
#   ???
# study = optuna.create_study(???)
# study.optimize(???)

### Solution

In [ ]:
EXECUTIONS_PER_TRIAL = 3


def objective(trial: optuna.Trial) -> float:
  units = trial.suggest_int("units", 32, 512, step=32)
  learning_rate = trial.suggest_categorical("learning_rate",
                                            [1e-2, 1e-3, 1e-4])

  # Chaque configuration est entraînée plusieurs fois, l'objectif étant la
  # moyenne des accuracies de validation obtenues
  val_accuracies = []
  for _ in range(EXECUTIONS_PER_TRIAL):
    model = nn.Sequential(nn.Linear(input_dim, units),
                          nn.ReLU(),
                          nn.Linear(units, nb_classes))
    classifier = MNISTClassifier(model,
                                 optimizer=torch.optim.Adam,
                                 learning_rate=learning_rate)
    train_loader, val_loader = get_dataloaders(X_train, y_train,
                                               batch_size=1500)
    trainer = lightning.Trainer(max_epochs=10,
                                accelerator="auto",
                                devices=1,
                                logger=False,
                                enable_checkpointing=False,
                                enable_progress_bar=False,
                                enable_model_summary=False)
    trainer.fit(classifier, train_loader, val_loader)
    val_accuracies.append(
        float(trainer.callback_metrics["val_accuracy"]))
  return sum(val_accuracies) / len(val_accuracies)


study = optuna.create_study(direction="maximize", study_name="mnist")

In [ ]:
study.optimize(objective, n_trials=5)

In [ ]:
print(study.trials_dataframe())
print(f"Meilleurs hyper-paramètres : {study.best_params}")
print(f"Meilleure accuracy de validation : {study.best_value}")

## Utilisation de TensorBoard pour la visualisation de métriques

Passer un [`TensorBoardLogger`](https://lightning.ai/docs/pytorch/stable/extensions/generated/lightning.pytorch.loggers.TensorBoardLogger.html) au `Trainer` dans son argument `logger` permet d'activer TensorBoard. On peut ensuite visualiser les entraînements directement dans Colab à l'aide de l'extension `tensorboard`.

In [ ]:
now = datetime.datetime.now().strftime("%Y-%m-%d-%H-%M-%S")

train_loader, val_loader = get_dataloaders(X_train, y_train, batch_size=3000)

deep_classifier = MNISTClassifier(build_deep_model(),
                                  optimizer=torch.optim.Adam,
                                  learning_rate=1e-3,
                                  weight_decay=0.01)

deep_trainer = lightning.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    logger=TensorBoardLogger("logs", name=f"adam-{now}"),
    enable_checkpointing=False)
deep_trainer.fit(deep_classifier, train_loader, val_loader)


# Même modèle mais avec sgd comme optimiseur
deep_classifier = MNISTClassifier(build_deep_model(),
                                  optimizer=torch.optim.SGD,
                                  learning_rate=0.01,
                                  weight_decay=0.01)

deep_trainer = lightning.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    logger=TensorBoardLogger("logs", name=f"sgd-{now}"),
    enable_checkpointing=False)
deep_trainer.fit(deep_classifier, train_loader, val_loader)

%reload_ext tensorboard
%tensorboard --logdir logs